# AOD product comparison — Vietnam maps

For each chosen (day, slot) show four panels side-by-side, all clipped to the same grid and overlaid with the GADM Vietnam boundary:

1. **Raw Himawari** (`Stage_A/gridded/…gridded_*.nc`) — L2 preferred, L3 fallback (you choose the rule in cell §2).
2. **Merged Stage A** (`Stage_A/merged/…merged_*.nc`, variable `AOD_merged`).
3. **RF gap-fill** (`Stage_B/output/rf/…aod_*.nc`, `aod_550nm`).
4. **ST kriging** (`Stage_B/output/st_kriging/…aod_*.nc`, `aod_550nm`).

Days are picked to span the Jan 2025 → Apr 2026 window across seasons; edit `DAYS`/`SLOTS_UTC` to change.

## 1. Setup

In [ ]:
from pathlib import Path
from datetime import datetime
import numpy as np
import xarray as xr
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

GRIDDED_DIR = Path('/home/slow_data/Air_Quality/Stage_A/gridded')
MERGED_DIR  = Path('/home/slow_data/Air_Quality/Stage_A/merged')
RF_DIR      = Path('/home/slow_data/Air_Quality/Stage_B/output/rf')
KRG_DIR     = Path('/home/slow_data/Air_Quality/Stage_B/output/st_kriging')
GADM_SHP    = Path('/home/work1/projects/Air_Quality/GADM_Vietnam/gadm41_VNM_1.shp')  # provinces

# Grid geometry — matches Stage A config (310 lat × 160 lon, 0.05°)
LAT0, LAT1 = 23.475, 8.025
LON0, LON1 = 102.025, 109.975
EXTENT = (LON0 - 0.025, LON1 + 0.025, LAT1 - 0.025, LAT0 + 0.025)

# Load Vietnam boundary once
vn = gpd.read_file(GADM_SHP)
print(f'GADM provinces: {len(vn)}   grid extent: {EXTENT}')

## 2. Days & slots to visualise

Four days spanning the requested window, three slots each (morning / noon / afternoon UTC — Vietnam is UTC+7, so 02:30/04:30/07:30 UTC = 09:30/11:30/14:30 local).

In [ ]:
DAYS = [
    '2025-02-15',   # NE-monsoon dry-haze season, North Vietnam
    '2025-06-15',   # pre-monsoon biomass-burning peak
    '2025-09-15',   # wet season, cleaner air, more cloud gaps
    '2026-01-15',   # winter 2026, dry-haze again
]
SLOTS_UTC = ['0230', '0430', '0730']   # 09:30, 11:30, 14:30 local
PRODUCTS  = ['raw_himawari', 'merged', 'rf', 'st_kriging']

## 3. Loaders — one function per product

Each returns a 2-D `numpy` array of AOD on the 310×160 grid, or `None` if the file is absent. The raw Himawari panel uses `AOD_himawari_l2` only (L3 is ignored).

In [ ]:
def _open(path):
    return xr.open_dataset(path) if path.exists() else None

def raw_himawari_aod(date_str: str, slot: str):
    """Return raw Himawari L2 AOD on the 310×160 grid for one slot."""
    y, m, d = date_str.split('-')
    path = GRIDDED_DIR / y / m / d / f'gridded_{y}{m}{d}_{slot}.nc'
    ds = _open(path)
    if ds is None:
        return None
    arr = ds['AOD_himawari_l2'].values.astype('float32')
    ds.close()
    return arr

In [ ]:
def merged_aod(date_str, slot):
    y, m, d = date_str.split('-')
    path = MERGED_DIR / y / m / d / f'merged_{y}{m}{d}_{slot}.nc'
    ds = _open(path)
    if ds is None:
        return None
    arr = ds['AOD_merged'].values.astype('float32')
    ds.close()
    return arr

def rf_aod(date_str, slot):
    y, m, d = date_str.split('-')
    path = RF_DIR / y / m / d / f'aod_{y}{m}{d}_{slot}.nc'
    ds = _open(path)
    if ds is None:
        return None
    arr = ds['aod_550nm'].values.astype('float32')
    ds.close()
    return arr

def krg_aod(date_str, slot):
    y, m, d = date_str.split('-')
    path = KRG_DIR / y / m / d / f'aod_{y}{m}{d}_{slot}.nc'
    ds = _open(path)
    if ds is None:
        return None
    arr = ds['aod_550nm'].values.astype('float32')
    ds.close()
    return arr

LOADERS = {
    'raw_himawari': raw_himawari_aod,
    'merged'      : merged_aod,
    'rf'          : rf_aod,
    'st_kriging'  : krg_aod,
}

## 4. Plot one day (3 slots × 4 products)

In [ ]:
# Shared colour scale so panels on a day are directly comparable.
VMIN, VMAX = 0.0, 1.5
CMAP = 'turbo'

# Panel dims chosen so the fig fits a thesis text block (~6.7" wide) with 4 cols.
# We use aspect='auto' below so the map stretches to fill the whole panel —
# at Vietnam's latitude the resulting distortion (cos(15°) ≈ 0.97) is invisible.
LON_SPAN = EXTENT[1] - EXTENT[0]
LAT_SPAN = EXTENT[3] - EXTENT[2]
PANEL_W  = 1.65
PANEL_H  = PANEL_W * (LAT_SPAN / LON_SPAN)   # ≈ 3.2"

def plot_day(date_str, slots=SLOTS_UTC, products=PRODUCTS, save_path=None):
    nrows, ncols = len(slots), len(products)
    fig, axes = plt.subplots(nrows, ncols,
                             figsize=(PANEL_W * ncols + 0.6, PANEL_H * nrows + 0.4))
    if nrows == 1:
        axes = np.array([axes])

    # Tight margins — no wasted space around or between panels.
    fig.subplots_adjust(left=0.05, right=0.90, bottom=0.03, top=0.94,
                        wspace=0.03, hspace=0.05)

    im = None
    for r, slot in enumerate(slots):
        for c, prod in enumerate(products):
            ax = axes[r, c]
            try:
                arr = LOADERS[prod](date_str, slot)
            except NotImplementedError:
                ax.set_title(f'{prod}\n(not implemented)', fontsize=7)
                ax.set_xticks([]); ax.set_yticks([])
                continue
            if arr is None:
                ax.set_title(f'{prod}\n(missing)', fontsize=7)
                ax.set_xticks([]); ax.set_yticks([])
                continue

            # aspect='auto' → map fills the whole axes box, zero internal whitespace.
            im = ax.imshow(arr, extent=EXTENT, origin='upper',
                           vmin=VMIN, vmax=VMAX, cmap=CMAP,
                           interpolation='nearest', aspect='auto')
            vn.boundary.plot(ax=ax, color='white', linewidth=0.35, alpha=0.9)
            ax.set_xlim(EXTENT[0], EXTENT[1])
            ax.set_ylim(EXTENT[2], EXTENT[3])

            if r == 0:
                ax.set_title(prod, fontsize=8, pad=2)
            if c == 0:
                ax.set_ylabel(f'{slot} UTC', fontsize=8)
                ax.tick_params(axis='y', labelsize=6)
            else:
                ax.set_yticklabels([])
            if r == nrows - 1:
                ax.tick_params(axis='x', labelsize=6)
            else:
                ax.set_xticklabels([])

    if im is not None:
        cax = fig.add_axes([0.915, 0.05, 0.015, 0.88])
        cbar = fig.colorbar(im, cax=cax)
        cbar.set_label('AOD @ 550 nm', fontsize=8)
        cbar.ax.tick_params(labelsize=7)
    fig.suptitle(date_str, fontsize=10, y=0.985)
    if save_path:
        fig.savefig(save_path, dpi=200, bbox_inches='tight')
    plt.show()

## 5. Render every chosen day

In [ ]:
for day in DAYS:
    plot_day(day)